# Train CIFAR-100 with PyTorch on Kaggle

This notebook trains an ImageNet-pretrained TorchVision MobileNetV2 using `CIFAR100/CIFAR100/{TRAIN,VALID,TEST}`. It uses a Kaggle GPU with mixed precision and saves the best checkpoint, class names, and test metrics under `/kaggle/working`.


In [ ]:
from pathlib import Path
import json
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU found. In Kaggle, select Settings > Accelerator > GPU, then restart the session.')
device = torch.device('cuda:0')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

IMAGE_SIZE = 32
BATCH_SIZE = 128
EPOCHS = 20
PATIENCE = 3
OUTPUT_DIR = Path('/kaggle/working')


## Find the dataset and create data loaders

Each split must have one class folder per category. Validation and test labels use the same class-to-index mapping as training.


In [ ]:
input_dir = Path('/kaggle/input')
split_dirs = [p.parent for p in input_dir.rglob('TRAIN')
              if p.is_dir() and (p.parent / 'VALID').is_dir()
              and (p.parent / 'TEST').is_dir()]
if not split_dirs:
    raise FileNotFoundError('Could not find CIFAR100/CIFAR100/{TRAIN,VALID,TEST} under /kaggle/input')
DATA_DIR = split_dirs[0]
print('Using dataset:', DATA_DIR)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

train_data = datasets.ImageFolder(DATA_DIR / 'TRAIN', transform=train_transform)
class_names = train_data.classes
if len(class_names) != 100:
    raise ValueError(f'Expected 100 class folders, found {len(class_names)}')
valid_data = datasets.ImageFolder(DATA_DIR / 'VALID', transform=eval_transform)
test_data = datasets.ImageFolder(DATA_DIR / 'TEST', transform=eval_transform)
for split_name, split_data in [('VALID', valid_data), ('TEST', test_data)]:
    if split_data.class_to_idx != train_data.class_to_idx:
        raise ValueError(f'{split_name} class folders do not match TRAIN')
print(f'Classes: {len(class_names)} | train: {len(train_data)} | valid: {len(valid_data)} | test: {len(test_data)}')

loader_args = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_data, shuffle=True, **loader_args)
valid_loader = DataLoader(valid_data, shuffle=False, **loader_args)
test_loader = DataLoader(test_data, shuffle=False, **loader_args)


## Build pretrained MobileNetV2

The ImageNet feature extractor is frozen; only the new 100-class classifier is trained. CIFAR images remain at 32×32 to keep training fast.


In [ ]:
weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)
for parameter in model.features.parameters():
    parameter.requires_grad = False
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler('cuda')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


## Train and save the best checkpoint

Mixed precision is enabled on CUDA. The notebook monitors validation accuracy, stops early when it stops improving, and then evaluates the best checkpoint on `TEST`.


In [ ]:
def run_epoch(loader, training):
    model.train(training)
    if training:
        # Keep the frozen feature extractor's batch-normalization statistics fixed.
        model.features.eval()
    total_loss = torch.zeros((), device=device)
    total_correct = torch.zeros((), device=device)
    total_items = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.detach() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum()
        total_items += images.size(0)
    return (total_loss / total_items).item(), (total_correct / total_items).item()

checkpoint_path = OUTPUT_DIR / 'cifar100_mobilenetv2_pytorch.pth'
best_accuracy = 0.0
best_state = None
epochs_without_improvement = 0
history = []
for epoch in range(EPOCHS):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    valid_loss, valid_accuracy = run_epoch(valid_loader, training=False)
    history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'train_accuracy': train_accuracy,
                    'valid_loss': valid_loss, 'valid_accuracy': valid_accuracy})
    print(f"Epoch {epoch + 1}/{EPOCHS} - train loss: {train_loss:.4f}, train acc: {train_accuracy:.4f}, "
          f"valid loss: {valid_loss:.4f}, valid acc: {valid_accuracy:.4f}")
    if valid_accuracy > best_accuracy:
        best_accuracy = valid_accuracy
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        torch.save({'model_state_dict': best_state, 'class_names': class_names,
                    'architecture': 'mobilenet_v2'}, checkpoint_path)
        epochs_without_improvement = 0
        print('Saved best checkpoint:', checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('Early stopping.')
            break

model.load_state_dict(best_state)
test_loss, test_accuracy = run_epoch(test_loader, training=False)
(OUTPUT_DIR / 'class_names.json').write_text(json.dumps(class_names, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'pytorch_metrics.json').write_text(json.dumps({
    'model': 'mobilenet_v2', 'test_loss': test_loss, 'test_accuracy': test_accuracy,
    'best_validation_accuracy': best_accuracy, 'history': history,
}, indent=2), encoding='utf-8')
print(f'Test loss: {test_loss:.4f} | test accuracy: {test_accuracy:.4f}')
print('Saved checkpoint:', checkpoint_path)
